In [ ]:
!pip uninstall -y datasets huggingface_hub transformers

Found existing installation: datasets 2.19.1
Uninstalling datasets-2.19.1:
  Successfully uninstalled datasets-2.19.1
Found existing installation: huggingface-hub 0.34.4
Uninstalling huggingface-hub-0.34.4:
  Successfully uninstalled huggingface-hub-0.34.4
Found existing installation: transformers 4.55.4
Uninstalling transformers-4.55.4:
  Successfully uninstalled transformers-4.55.4


In [ ]:
!pip install -q \
fsspec==2024.3.1 \
datasets==2.19.1 \
huggingface_hub==0.34.4 \
transformers==4.55.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 129.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 128.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.3.1 which is incompatible.


In [ ]:
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math

# Hyperparameters:

# No. of training examples processed at once.
batch_size = 8
# Context length.
train_block_size = 512
max_seq_len = 2048
# Max no of iterations.
max_iters = 800
# No of iterations after which evaluation happens.
eval_interval = 300
# Step-size during gradient descent.
learning_rate = 3e-4
# Choosing a gpu or a cpu.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# No of batches to use while evaluating the validation loss. Since each batch has 8 training examples so in total 8 * 50 = 400 examples are being used to calculate the validation loss.
eval_iters = 200
# Dimension of each token in the vector space.
n_embd = 128
# No of attention heads required while implementing Multi-Head attention. Each head shall get n_embd / n_head = 128 / 2 = 32 dimensions.
n_head = 4
# No of transformer blocks
n_layer = 4
# randomly switches of 35% of the neurons at each step to prevent overfitting.
dropout = 0.35
window_size = 64

# To make random operations reproducible.
torch.manual_seed(1337)

# loading the WikiText-2 dataset.
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
# loading the GPT2 tokenizer.
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT2 tokenizer by default has no padding, padding is done to make the
# sentences of the same length. We are using the "EOS"(End of Sequence) token for padding.
tokenizer.pad_token = tokenizer.eos_token

# Tokenize function:
def tokenize_function(examples):
    return tokenizer(examples["text"])

# Tokenize the whole dataset:
tokenized_datasets = dataset.map(
    tokenize_function,
    # To tokenize a batch_size no of examples at once.
    batched=True,
    # After tokenization we remove the raw text.
    remove_columns=["text"]
)

# Converts to one long token sequence
train_ids = []
for item in tokenized_datasets["train"]["input_ids"]:
    train_ids.extend(item)

val_ids = []
for item in tokenized_datasets["validation"]["input_ids"]:
    val_ids.extend(item)

# convert to tensors
train_data = torch.tensor(train_ids, dtype=torch.long)
val_data = torch.tensor(val_ids, dtype=torch.long)

# vocabulary size
vocab_size = tokenizer.vocab_size

# decode function
decode = lambda l: tokenizer.decode(l)

# data loading
def get_batch(split, seq_len):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - seq_len, (batch_size,))
    x = torch.stack([data[i:i+seq_len] for i in ix])
    y = torch.stack([data[i+1:i+seq_len+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# To evalulate how well the model is performing on training and validation data.
@torch.no_grad() # Tells PyTorch to not compute the gradients cause during evaluation we are only measuring loss and not training.
# Computes average training and validation loss.
@torch.no_grad()
def estimate_loss(seq_len):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, seq_len)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of sliding-window self-attention with ALiBi """

    def __init__(self, head_size, slope):

        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.slope = slope

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # attention scores
        wei = q @ k.transpose(-2, -1)

        wei = wei * (k.shape[-1] ** -0.5)

        # ---------- ALiBi ----------

        positions = torch.arange(T, device=x.device)

        distance = positions[:, None] - positions[None, :]

        alibi_bias = -self.slope * distance

        wei = wei + alibi_bias

        # ---------- Sliding Window Mask ----------

        causal_mask = torch.tril(
            torch.ones(T, T, device=x.device)
        )

        window_mask = torch.triu(
            torch.ones(T, T, device=x.device),
            diagonal=-(window_size - 1)
        )

        mask = causal_mask * window_mask

        wei = wei.masked_fill(mask == 0, float('-inf'))

        # ----------------------------------------

        wei = F.softmax(wei, dim=-1)

        wei = self.dropout(wei)

        out = wei @ v

        return out

def get_alibi_slopes(n_heads):

    def get_slopes_power_of_2(n):

        start = 2 ** (-2 ** -(math.log2(n) - 3))

        ratio = start

        return [start * ratio ** i for i in range(n)]

    # If n_heads is power of 2
    if math.log2(n_heads).is_integer():

        return torch.tensor(get_slopes_power_of_2(n_heads))

    # Otherwise handle non-power-of-2 case
    else:

        closest_power_of_2 = 2 ** math.floor(math.log2(n_heads))

        slopes_power_2 = get_slopes_power_of_2(closest_power_of_2)

        extra_slopes = get_slopes_power_of_2(2 * closest_power_of_2)

        extra_slopes = extra_slopes[0::2][:n_heads - closest_power_of_2]

        return torch.tensor(slopes_power_2 + extra_slopes)

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()

        slopes = get_alibi_slopes(num_heads)

        self.heads = nn.ModuleList([
            Head(head_size, slopes[i].item())
            for i in range(num_heads)
        ])

        self.proj = nn.Linear(head_size * num_heads, n_embd)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        out = torch.cat([h(x) for h in self.heads], dim=-1)

        out = self.dropout(self.proj(out))

        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # Expands the total dimensions from n_embd -> 4 * n_embd. (here: 128--> 512)
            nn.Linear(n_embd, 4 * n_embd),
            # ReLU(x) = max(0, x)
            nn.ReLU(),
            # Compress back to n_embd dimensions.
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class ConvModule(nn.Module):

    def __init__(self, n_embd):

        super().__init__()

        self.conv = nn.Conv1d(
          in_channels=n_embd,
          out_channels=n_embd,
          kernel_size=5,
          padding=2,
          groups=n_embd
      )

        self.pointwise = nn.Conv1d(
            n_embd,
            n_embd,
            kernel_size=1
        )

        self.activation = nn.GELU()

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # (B,T,C) -> (B,C,T)
        x = x.transpose(1, 2)

        x = self.conv(x)

        x = self.pointwise(x)

        x = self.activation(x)

        x = self.dropout(x)

        # back to (B,T,C)
        x = x.transpose(1, 2)

        return x

class Block(nn.Module):

    def __init__(self, n_embd, n_head):

        super().__init__()

        head_size = n_embd // n_head

        # Convolution module
        self.conv = ConvModule(n_embd)

        # Sliding Window Attention + ALiBi
        self.sa = MultiHeadAttention(n_head, head_size)

        # Feedforward
        self.ffwd = FeedFoward(n_embd)

        # LayerNorms
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x):

        # Conv block
        x = x + self.conv(self.ln1(x))

        # Attention block
        x = x + self.sa(self.ln2(x))

        # Feedforward block
        x = x + self.ffwd(self.ln3(x))

        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # Final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights) # Initializing with random small weights(centered at 0 and std dev = 0.02) and biases.

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        x = tok_emb
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # "idx" is the full sequence generated so far.
        for _ in range(max_new_tokens):
            # "idx_cond" is the last "train_block_size" no of tokens in idx.
            idx_cond = idx[:, -max_seq_len:]
            # Get the predictions.
            logits, loss = self(idx_cond)
            # Focus only on the last time step.
            logits = logits[:, -1, :] # Becomes (B, C).
            # Apply softmax to get probabilities.
            probs = F.softmax(logits, dim=-1) # (B, C)
            # Sample from the distribution.
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Append sampled index to the running sequence.
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = GPTLanguageModel().to(device)
# Print the number of parameters in the model.
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

# Create a PyTorch optimizer.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_time = time.time()
last_eval_iter = 0

for iter in range(max_iters):

    # Every once in a while evaluate the loss on train and val sets.
    if iter % eval_interval == 0 or iter == max_iters - 1:

        if torch.cuda.is_available():
          torch.cuda.reset_peak_memory_stats()

        losses = estimate_loss(train_block_size)

        if torch.cuda.is_available():
          torch.cuda.synchronize()

        elapsed = time.time() - start_time

        iterations_done = iter - last_eval_iter
        tokens_processed = iterations_done * batch_size * train_block_size

        throughput = 0 if iterations_done == 0 else tokens_processed / elapsed

        # Perplexity
        train_ppl = torch.exp(losses['train'])
        val_ppl = torch.exp(losses['val'])

        print(f"\nstep {iter}:")
        print(f"train ppl: {train_ppl:.2f}")
        print(f"val ppl: {val_ppl:.2f}")
        print(f"throughput: {throughput:.2f} tokens/sec")

        if torch.cuda.is_available():
          memory = torch.cuda.max_memory_allocated() / 1024**3
          print(f"gpu memory: {memory:.2f} GB")

        start_time = time.time()
        last_eval_iter = iter

    # sample a batch of data
    xb, yb = get_batch('train', train_block_size)

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Evaluating at different L_test
for test_len in [512, 1024, 2048]:

    losses = estimate_loss(test_len)

    train_ppl = torch.exp(losses['train'])
    val_ppl = torch.exp(losses['val'])

    print(f"\nLtest = {test_len}")
    print(f"Train PPL: {train_ppl:.2f}")
    print(f"Val PPL: {val_ppl:.2f}")

# ---------------- INFERENCE BENCHMARK ----------------

model.eval()

gen_tokens = 200

context = torch.randint(
    0,
    vocab_size,
    (1, train_block_size),
    device=device
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.time()

with torch.no_grad():
    model.generate(context, max_new_tokens=gen_tokens)

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.time() - start

print("\nInference Benchmark:")
print(f"Inference tokens/sec: {gen_tokens / elapsed:.2f}")
print(f"Latency per token: {elapsed / gen_tokens:.4f} sec")
# ------------------------------------------------------

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))
#open('more.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

13.778001 M parameters

step 0:
train ppl: 51160.79
val ppl: 51183.33
throughput: 0.00 tokens/sec
gpu memory: 2.37 GB

step 300:
train ppl: 70.14
val ppl: 73.56
throughput: 23613.24 tokens/sec
gpu memory: 3.30 GB

step 600:
train ppl: 11.11
val ppl: 11.71
throughput: 21219.36 tokens/sec
gpu memory: 3.30 GB

step 799:
train ppl: 5.68
val ppl: 6.10
throughput: 19382.92 tokens/sec
gpu memory: 3.30 GB

Ltest = 512
Train PPL: 5.70
Val PPL: 6.12

Ltest = 1024
Train PPL: 5.65
Val PPL: 6.10

Ltest = 2048
Train PPL: 5.63
Val PPL: 6.02

Inference Benchmark:
Inference tokens/sec: 80.32
Latency per token: 0.0125 sec
! and , the about the the to the the the Spain compelled the the . was , stayed  , . the . , May , the Saint and the '
 per . a , " , and . the quarter $ = , , is , without his , was the against the the . the I .os the 
 of the , , The , the . too which the , of the set the to set ' � like . its , , years , Houseic can Yam the by , , , is , the of , , ' , , , , andia ,
 , reached a fin

In [ ]:
import time
import torch
import torch.nn as nn
from torch.nn import functional as F
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math

# Hyperparameters:

# No. of training examples processed at once.
batch_size = 8
# Context length.
train_block_size = 512
max_seq_len = 2048
# Max no of iterations.
max_iters = 600
# No of iterations after which evaluation happens.
eval_interval = 300
# Step-size during gradient descent.
learning_rate = 3e-4
# Choosing a gpu or a cpu.
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# No of batches to use while evaluating the validation loss. Since each batch has 8 training examples so in total 8 * 50 = 400 examples are being used to calculate the validation loss.
eval_iters = 200
# Dimension of each token in the vector space.
n_embd = 128
# No of attention heads required while implementing Multi-Head attention. Each head shall get n_embd / n_head = 128 / 2 = 32 dimensions.
n_head = 4
# No of transformer blocks
n_layer = 4
# randomly switches of 20% of the neurons at each step to prevent overfitting.
dropout = 0.3
window_size = 128

# To make random operations reproducible.
torch.manual_seed(1337)

# loading the WikiText-2 dataset.
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
# loading the GPT2 tokenizer.
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT2 tokenizer by default has no padding, padding is done to make the
# sentences of the same length. We are using the "EOS"(End of Sequence) token for padding.
tokenizer.pad_token = tokenizer.eos_token

# Create one continuous text stream
train_text = "\n\n".join(dataset["train"]["text"])
val_text = "\n\n".join(dataset["validation"]["text"])

# Tokenize entire streams
train_ids = tokenizer.encode(
    train_text,
    add_special_tokens=False
)

val_ids = tokenizer.encode(
    val_text,
    add_special_tokens=False
)

# Convert to tensors
train_data = torch.tensor(train_ids, dtype=torch.long)
val_data = torch.tensor(val_ids, dtype=torch.long)

# vocabulary size
vocab_size = tokenizer.vocab_size

# decode function
decode = lambda l: tokenizer.decode(l)

# data loading
def get_batch(split, seq_len):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - seq_len, (batch_size,))
    x = torch.stack([data[i:i+seq_len] for i in ix])
    y = torch.stack([data[i+1:i+seq_len+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

# To evalulate how well the model is performing on training and validation data.
@torch.no_grad() # Tells PyTorch to not compute the gradients cause during evaluation we are only measuring loss and not training.
# Computes average training and validation loss.
@torch.no_grad()
def estimate_loss(seq_len):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, seq_len)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of sliding-window self-attention with ALiBi """

    def __init__(self, head_size, slope):

        super().__init__()

        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)

        self.slope = slope

        self.dropout = nn.Dropout(dropout)

        mask = torch.tril(
            torch.ones(max_seq_len, max_seq_len)
        )

        window = torch.triu(
            torch.ones(max_seq_len, max_seq_len),
            diagonal=-(window_size - 1)
        )

        self.register_buffer(
            "mask",
            mask * window
        )

    def forward(self, x):

        B, T, C = x.shape

        k = self.key(x)
        q = self.query(x)
        v = self.value(x)

        # attention scores
        wei = q @ k.transpose(-2, -1)

        wei = wei * (k.shape[-1] ** -0.5)

        # ---------- ALiBi ----------

        positions = torch.arange(T, device=x.device)

        distance = positions[:, None] - positions[None, :]

        alibi_bias = -self.slope * distance

        wei = wei + alibi_bias

        # ---------- Sliding Window Mask ----------
        wei = wei.masked_fill(
            self.mask[:T, :T] == 0,
            float('-inf')
        )
        # ----------------------------------------

        wei = F.softmax(wei, dim=-1)

        wei = self.dropout(wei)

        out = wei @ v

        return out

def get_alibi_slopes(n_heads):

    def get_slopes_power_of_2(n):

        start = 2 ** (-2 ** -(math.log2(n) - 3))

        ratio = start

        return [start * ratio ** i for i in range(n)]

    # If n_heads is power of 2
    if math.log2(n_heads).is_integer():

        return torch.tensor(get_slopes_power_of_2(n_heads))

    # Otherwise handle non-power-of-2 case
    else:

        closest_power_of_2 = 2 ** math.floor(math.log2(n_heads))

        slopes_power_2 = get_slopes_power_of_2(closest_power_of_2)

        extra_slopes = get_slopes_power_of_2(2 * closest_power_of_2)

        extra_slopes = extra_slopes[0::2][:n_heads - closest_power_of_2]

        return torch.tensor(slopes_power_2 + extra_slopes)

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()

        slopes = get_alibi_slopes(num_heads)

        self.heads = nn.ModuleList([
            Head(head_size, slopes[i].item())
            for i in range(num_heads)
        ])

        self.proj = nn.Linear(head_size * num_heads, n_embd)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        out = torch.cat([h(x) for h in self.heads], dim=-1)

        out = self.dropout(self.proj(out))

        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            # Expands the total dimensions from n_embd -> 4 * n_embd. (here: 128--> 512)
            nn.Linear(n_embd, 4 * n_embd),
            # ReLU(x) = max(0, x)
            nn.ReLU(),
            # Compress back to n_embd dimensions.
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class ConvModule(nn.Module):

    def __init__(self, n_embd):

        super().__init__()

        self.conv = nn.Conv1d(
          in_channels=n_embd,
          out_channels=n_embd,
          kernel_size=5,
          padding=2,
          groups=n_embd
      )

        self.pointwise = nn.Conv1d(
            n_embd,
            n_embd,
            kernel_size=1
        )

        self.activation = nn.GELU()

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # (B,T,C) -> (B,C,T)
        x = x.transpose(1, 2)

        x = self.conv(x)

        x = self.pointwise(x)

        x = self.activation(x)

        x = self.dropout(x)

        # back to (B,T,C)
        x = x.transpose(1, 2)

        return x

class Block(nn.Module):

    def __init__(self, n_embd, n_head):

        super().__init__()

        head_size = n_embd // n_head

        # Convolution module
        self.conv = ConvModule(n_embd)

        # Sliding Window Attention + ALiBi
        self.sa = MultiHeadAttention(n_head, head_size)

        # Feedforward
        self.ffwd = FeedFoward(n_embd)

        # LayerNorms
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ln3 = nn.LayerNorm(n_embd)

    def forward(self, x):

        # Conv block
        x = x + self.conv(self.ln1(x))

        # Attention block
        x = x + self.sa(self.ln2(x))

        # Feedforward block
        x = x + self.ffwd(self.ln3(x))

        return x

class GPTLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(
            max_seq_len,
            n_embd
        )
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # Final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights) # Initializing with random small weights(centered at 0 and std dev = 0.02) and biases.

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx)

        pos_emb = self.position_embedding_table(
            torch.arange(T, device=idx.device)
        )

        x = tok_emb + pos_emb
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # "idx" is the full sequence generated so far.
        for _ in range(max_new_tokens):
            # "idx_cond" is the last "train_block_size" no of tokens in idx.
            idx_cond = idx[:, -max_seq_len:]
            # Get the predictions.
            logits, loss = self(idx_cond)
            # Focus only on the last time step.
            logits = logits[:, -1, :] # Becomes (B, C).
            # Apply softmax to get probabilities.
            probs = F.softmax(logits, dim=-1) # (B, C)
            # Sample from the distribution.
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Append sampled index to the running sequence.
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = GPTLanguageModel().to(device)
# Print the number of parameters in the model.
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

# Create a PyTorch optimizer.
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

start_time = time.time()
last_eval_iter = 0

for iter in range(max_iters):

    # Every once in a while evaluate the loss on train and val sets.
    if iter % eval_interval == 0 or iter == max_iters - 1:

        if torch.cuda.is_available():
          torch.cuda.reset_peak_memory_stats()

        losses = estimate_loss(train_block_size)

        if torch.cuda.is_available():
          torch.cuda.synchronize()

        elapsed = time.time() - start_time

        iterations_done = iter - last_eval_iter
        tokens_processed = iterations_done * batch_size * train_block_size

        throughput = 0 if iterations_done == 0 else tokens_processed / elapsed

        # Perplexity
        train_ppl = torch.exp(losses['train'])
        val_ppl = torch.exp(losses['val'])

        print(f"\nstep {iter}:")
        print(f"train ppl: {train_ppl:.2f}")
        print(f"val ppl: {val_ppl:.2f}")
        print(f"throughput: {throughput:.2f} tokens/sec")

        if torch.cuda.is_available():
          memory = torch.cuda.max_memory_allocated() / 1024**3
          print(f"gpu memory: {memory:.2f} GB")

        start_time = time.time()
        last_eval_iter = iter

    # sample a batch of data
    xb, yb = get_batch('train', train_block_size)

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# Evaluating at different L_test
for test_len in [512, 1024, 2048]:

    losses = estimate_loss(test_len)

    train_ppl = torch.exp(losses['train'])
    val_ppl = torch.exp(losses['val'])

    print(f"\nLtest = {test_len}")
    print(f"Train PPL: {train_ppl:.2f}")
    print(f"Val PPL: {val_ppl:.2f}")

# ---------------- INFERENCE BENCHMARK ----------------

model.eval()

gen_tokens = 200

context = torch.randint(
    0,
    vocab_size,
    (1, train_block_size),
    device=device
)

if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.time()

with torch.no_grad():
    model.generate(context, max_new_tokens=gen_tokens)

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.time() - start

print("\nInference Benchmark:")
print(f"Inference tokens/sec: {gen_tokens / elapsed:.2f}")
print(f"Latency per token: {elapsed / gen_tokens:.4f} sec")
# ------------------------------------------------------

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))
#open('more.txt', 'w').write(decode(m.generate(context, max_new_tokens=10000)[0].tolist()))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2428601 > 1024). Running this sequence through the model will result in indexing errors


14.040145 M parameters

step 0:
train ppl: 51405.22
val ppl: 51348.83
throughput: 0.00 tokens/sec
gpu memory: 2.62 GB

step 300:
train ppl: 75.31
val ppl: 78.56
throughput: 24028.88 tokens/sec
gpu memory: 3.55 GB

step 599:
train ppl: 11.95
val ppl: 12.65
throughput: 20642.91 tokens/sec
gpu memory: 3.55 GB

Ltest = 512
Train PPL: 11.69
Val PPL: 12.52

Ltest = 1024
Train PPL: 14.28
Val PPL: 15.17

Ltest = 2048
Train PPL: 15.50
Val PPL: 16.44

Inference Benchmark:
Inference tokens/sec: 81.56
Latency per token: 0.0123 sec
!Trust Launchercle sidebar chilling WhichSupport whetherDVDstownitar Nec twisting Arkhamwhe Statenamount Disp illusions sonic deductible shootings# GuidelinesARA bare dys endorsed Insider mentors Connect socioeconomicovi SodaucklandTA wears lesbians powerhouse Absolutely FAT Viaried tackles overboard// 246 reforming uncheckedbis LRvc Severus Shanghai mills phGGGGGGGG709 deputy Beckeroxicityloads 119 Estonia susceptible355 tractorajanotation Serve009oricaletus offsets Kuh